# Time Series Prediction

**数据探索 → 特征工程 → 模型构建 的完整流程**

- **整体探索**：关注长期趋势、异常值、特殊时期
- **分维度探索**：从长周期到短周期（年 → 月 → 日 → 小时），跨主体可标准化后对比
- **核心目的**：探索不同维度对预测变量的影响，为特征构建提供理论和数据支撑
- **模型构建**：记得关注数据探索中发现的所有特殊时段与特征

> 本 notebook 以电力负荷（energy load）类小时级时间序列为例，目标列名为 `amount`，索引为 `DatetimeIndex`。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import holidays
import statsmodels.api as sm
import xgboost as xgb
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

## 0. 载入数据

将你的数据载入为一个带有 `DatetimeIndex` 的 `DataFrame`，目标列命名为 `amount`。

下方提供一个**合成数据生成器**，便于在没有真实数据时端到端跑通整个流程（含长期趋势、年度季节性、周内效应、小时双峰、节假日效应，以及 2012 年的一段异常）。接入真实数据时，删除合成部分、保留 `df` 接口即可。

In [ ]:
# === 选项 A: 载入真实数据 ===
# df = pd.read_csv("your_data.csv", index_col=0, parse_dates=True)
# df = df.rename(columns={"<your_target_col>": "amount"})
# df = df[["amount"]].sort_index()

# === 选项 B: 合成数据（演示用，可删） ===
rng = np.random.default_rng(42)
idx = pd.date_range("2010-01-01", "2018-12-31 23:00", freq="h")
n = len(idx)

t = np.arange(n)
trend = 0.0008 * t                                   # 缓慢长期上升趋势
yearly = 80 * np.cos(2*np.pi*(idx.dayofyear/365.25)) # 冬夏高峰
weekly = -15 * idx.dayofweek.isin([5, 6]).astype(int) # 周末偏低
hourly = 40*np.sin(2*np.pi*(idx.hour-3)/24) + 25*np.sin(2*np.pi*(idx.hour-3)/12)  # 双峰
us_hol = holidays.US()
hol = np.array([-30 if d.date() in us_hol else 0 for d in idx])
noise = rng.normal(0, 12, n)

amount = 500 + trend*60 + yearly + weekly + hourly + hol + noise

df = pd.DataFrame({"amount": amount}, index=idx)
df.index.name = "datetime"

# 注入 2012 年的一段下行异常（模拟设备/数据问题），供 2.1 异常处理演示
anom = (df.index.year == 2012) & (df.index.month == 6)
df.loc[anom, "amount"] -= 200

df.head()

# 1. Data Exploration

先做一些与时间相关的基础 features，方便后续在不同维度上分析（从长周期慢慢到短周期）：

- Year / Month / Day of Week / Holiday / Hour
- 周期性的 sin/cos 编码
- 是否周末、是否工作时段

In [ ]:
df["hour"] = df.index.hour
df["month"] = df.index.month
df['quarter'] = df.index.quarter
df['year'] = df.index.year
df['dayofweek'] = df.index.dayofweek
df['dayofmonth'] = df.index.day
df['dayofyear'] = df.index.dayofyear

df["is_weekend"] = df['dayofweek'].isin([5, 6]).astype(int)
us_holiday = holidays.US()
df['is_holiday'] = df.index.map(lambda x: 1 if x in us_holiday else 0)

df['hour_sin'] = np.sin(2 * np.pi * df["hour"] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df["hour"] / 24)

df['is_working_hour'] = df["hour"].between(8, 18).astype(int)
df.head()

## 1.1 Full History — 描述性统计

查看数据缺失、异常值等并决定是否处理。`describe` 函数在标准描述统计基础上补充了分位点、偏度 (skew)、峰度 (kurt)。

In [ ]:
def describe(df: pd.DataFrame) -> pd.DataFrame:
    desc = df.describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
    moments = pd.DataFrame(
        [
            df.skew(axis=0),
            df.kurt(axis=0)
        ], index=["skew", "kurt"], columns=df.columns
    )
    return pd.concat([desc, moments], axis=0)

describe(df[["amount"]])

In [ ]:
# 缺失值检查
df.isna().sum()

## 1.2 Yearly Trends

**主要目的**：查看数据中是否具有长期趋势。

**方法一**：降频成更低频率（日），用 mean ± std 做类 Bollinger Bands plot，辅以长期均线（365D rolling mean），查看是否存在长期趋势。

In [ ]:
daily_avg = df["amount"].resample("D", label="right", closed="right").mean()
daily_std = df["amount"].resample("D", label="right", closed="right").std()
daily_avg_rolling_mean_year = daily_avg.rolling(window=365, center=True).mean()

plt.figure(figsize=(15, 5))
plt.plot(daily_avg, label="Daily Average Load")
plt.fill_between(
    daily_avg.index,
    daily_avg - daily_std,
    daily_avg + daily_std,
    alpha=0.3,
    label="Daily Variability."
)
plt.plot(daily_avg_rolling_mean_year, color='red', linewidth=2, label='365D Rolling Mean (Trend)')
plt.title("Daily Average Energy Consumption with Variability")
plt.legend()
plt.show()

**方法二**：按年 groupby 做描述性统计 + box plot，查看分位点随年份的变化。

In [ ]:
year_desc = df[["amount"]].groupby(df.index.year).apply(describe, include_groups=False).unstack()

fig, ax = plt.subplots(figsize=(15, 6))
sns.boxplot(data=df, x=df.index.year, y='amount')
ax.set_title('Amount Distribution by Year')
plt.show()

year_desc

**方法三**：计算整体方差 vs 各年份内部方差（取平均）。

- 若两者差别不大 → 绝大部分波动发生在**年度内部**（季节性、小时性等更低频波动），跨年度没有特别大的差异。
- 若差别大 → 存在年度间的分布漂移 / 趋势。

**基于图像和数据给我们的启示**：
1. 长期趋势的判断
2. 是否有异常值
3. 历史上是否发生过分布上的变化、漂移
4. 是否需要构造其他衍生 features
5. 判断是否有异常情况需要处理

In [ ]:
overall_var = df["amount"].var()
within_year_var = df.groupby("year")["amount"].var().mean()

print(f"整体方差 (overall variance):        {overall_var:,.2f}")
print(f"各年份内部方差均值 (within-year avg): {within_year_var:,.2f}")
print(f"比值 within / overall:              {within_year_var / overall_var:.3f}")
print("\n比值接近 1 → 波动主要来自年内（季节/小时）；明显 < 1 → 存在跨年度趋势/漂移。")

## 1.3 Monthly Trends

**方法一**：按 month groupby 做描述性统计 + box plot，查看季节性规律。

In [ ]:
month_desc = df[['amount']].groupby(df.index.month).apply(describe, include_groups=False).unstack()

fig, ax = plt.subplots(figsize=(15, 6))
sns.boxplot(data=df, x=df.index.month, y='amount')
ax.set_title('Amount Distribution by Month')
plt.show()

month_desc

**方法二**：区分不同年度对各月份统计，查看年内季节性趋势是否跨年度稳定。曲线形状高度一致 → 季节性非常稳定。

In [ ]:
monthly_trend = df.groupby(['year', 'month'])['amount'].mean().unstack(level=0)

plt.figure(figsize=(12, 6))
plt.plot(monthly_trend, alpha=0.5, linewidth=1)
plt.plot(monthly_trend.mean(axis=1), color='black', linewidth=3, label='Overall Mean')
plt.title('Seasonal Plot: Monthly Consumption by Year')
plt.xlabel('Month')
plt.ylabel('Average Amount')
plt.xticks(range(1, 13))
plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
plt.grid(True, alpha=0.3)
plt.show()

**方法三**：按年度 × 月份 pivot table 画 heatmap，快速识别某些特定年份是否出现偏离常规的"极热/极冷"月份；观察高峰、低谷期是否随时间变长/变短。

In [ ]:
pivot_table = df.pivot_table(index='year', columns='month', values='amount', aggfunc='mean')

plt.figure(figsize=(14, 8))
sns.heatmap(pivot_table, annot=True, fmt=".0f", cmap='YlOrRd', cbar_kws={'label': 'Average Amount'})
plt.title('Monthly Average Consumption Heatmap')
plt.xlabel('Month')
plt.ylabel('Year')
plt.show()

**方法四**：每月统计均值、标准差，计算月度变异系数 (Coefficient of Variation, $CV = \sigma / \mu$)。CV 越大 → 越不容易预测。

In [ ]:
monthly_stats = df.groupby('month')['amount'].agg(['mean', 'std'])
monthly_stats['cv'] = monthly_stats['std'] / monthly_stats['mean']

plt.figure(figsize=(10, 5))
sns.barplot(x=monthly_stats.index, y=monthly_stats['cv'], hue=monthly_stats.index, legend=False)
plt.axhline(monthly_stats['cv'].mean(), color='red', linestyle='--', label='Average CV')
plt.title('Coefficient of Variation (CV) by Month')
plt.xlabel('Month')
plt.ylabel('CV (Standard Deviation / Mean)')
plt.legend()
plt.show()

monthly_stats

## 1.4 Weekly and Hourly Trends

**方法一**：按 weekday / hour groupby 做描述性统计 + box plot，查看是否存在小时级差异、工作日 vs 非工作日差异、早晚高峰。

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
sns.boxplot(data=df, x=df.index.dayofweek, y='amount')
ax.set_title('Amount Distribution by Day of Week')
plt.show()

fig, ax = plt.subplots(figsize=(15, 6))
sns.boxplot(data=df, x=df.index.hour, y='amount')
ax.set_title('Amount Distribution by Hour')
plt.show()

**方法二**：按 weekday × hour pivot table 画 heatmap，查看不同 weekday 内的小时级趋势。

In [ ]:
pivot_hour_day = df.pivot_table(index='dayofweek', columns='hour', values='amount', aggfunc='mean')

plt.figure(figsize=(15, 6))
sns.heatmap(pivot_hour_day, cmap='RdYlBu_r', annot=False)
plt.title('Heatmap: Energy Consumption by Day of Week and Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week (0=Mon, 6=Sun)')
plt.show()

**方法三**：按 hour 取 mean，区分工作日 / 非工作日，凸显差异（阴影为 ±1 标准差）。

In [ ]:
df['is_weekend_str'] = df['is_weekend'].map({1: 'Weekend', 0: 'Weekday'})

plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='hour', y='amount', hue='is_weekend_str', errorbar='sd')
plt.title('Daily Profile: Weekday vs Weekend')
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()

## 1.5 Time Series Analysis

**方法一**：ACF / PACF 分析，在特定 lag（如 24h、168h）处查看规律。红线标出每天 24h 的倍数。

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axs = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
plot_acf(df['amount'], lags=170, ax=axs[0])
plot_pacf(df['amount'], lags=170, ax=axs[1], method='ywm')

for i in range(1, 8):
    for j in range(2):
        axs[j].axvline(x=i*24, color="red", alpha=0.5)
plt.show()

**方法二**：平稳性检验 (ADF) + 目标变量分布。

- 若一阶不平稳 → 考虑差分
- 若分布偏斜 → 考虑 log 变换

In [ ]:
from statsmodels.tsa.stattools import adfuller
import scipy.stats as stats

# 1. ADF 检验
result = adfuller(df['amount'].dropna())
print(f'ADF Statistic: {result[0]}')
print(f'p-value: {result[1]}')  # 若 < 0.05 则平稳

# 2. 目标变量分布
plt.figure(figsize=(10, 4))
sns.histplot(df['amount'], kde=True)
plt.title(f"Target Distribution (Skewness: {df['amount'].skew():.2f})")
plt.show()

# 2. Feature Engineering

## 2.1 异常值处理

在上面步骤中如发现异常部分，可做处理 —— 例如把异常值设为 NaN 再线性插值。这里针对演示数据中 **2012 年**的异常段进行处理。

In [ ]:
df = df.copy()
# --- Step 1: 异常值处理 (仅针对 2012) ---
mask_2012 = (df.index.year == 2012)
mean_2012 = df.loc[mask_2012, 'amount'].mean()
std_2012 = df.loc[mask_2012, 'amount'].std()

# 超过 2.5 倍标准差设为 NaN（此处仅处理向下异常）
# outlier_condition = (df['amount'] < (mean_2012 - 2.5 * std_2012)) | \
#                     (df['amount'] > (mean_2012 + 2.5 * std_2012))
outlier_condition = df['amount'] < (mean_2012 - 2.5 * std_2012)
cond = mask_2012 & outlier_condition
print(f"{cond.sum()} outliers in 2012")
df.loc[cond, 'amount'] = np.nan

# --- Step 2: 线性插值 ---
df['amount'] = df['amount'].interpolate(method='linear')

## 2.2 特征生成

把探索中发现的效应转化为特征：季节性月度效应、早晚高峰、是否工作日、是否假期等，并纳入交互特征。

- **线性模型**（如 LR）：偏好平滑特征 → 小时做 sin/cos；无法自动捕捉 `a*b` 交叉项 → 必要时手动构造。
- **非线性模型**（如 XGBoost）：各种特征都可以，包括类别索引、数值特征。
- 通用特征：lag、rolling mean、lag_diff。

In [ ]:
df['target_log'] = np.log1p(df['amount'])

# --- Step 4: 基础时间特征 & 关键指标 ---
df["hour"] = df.index.hour
df["month"] = df.index.month
df['quarter'] = df.index.quarter
df['year'] = df.index.year

df['dayofweek'] = df.index.dayofweek
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)

# 观察到的差异特征
df['is_peak_month'] = df['month'].isin([1, 7, 8]).astype(int)  # 夏冬高峰
df['is_peak_hour'] = df['hour'].between(18, 21).astype(int)    # 傍晚高峰

# 其他可能有效的特征
df['is_working_hour'] = df["hour"].between(8, 18).astype(int)
us_holiday = holidays.US()
df['is_holiday'] = df.index.map(lambda x: 1 if x in us_holiday else 0)

# --- Step 5: 交叉变量构建 ---
# 为 LR 准备的正余弦编码
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['hour_sin_weekend'] = df['hour_sin'] * df['is_weekend']
df['hour_cos_weekend'] = df['hour_cos'] * df['is_weekend']

# 为 XGBoost 准备的交叉索引
df['hour_day_idx'] = df['dayofweek'] * 24 + df['hour']

# --- Step 6: 滑动窗口特征 (捕捉近期趋势) ---
df['rolling_mean_24'] = df['target_log'].shift(1).rolling(window=24).mean()
df['lag_diff'] = df['target_log'].shift(1) - df['target_log'].shift(2)

# --- Step 7: 滞后特征 (在 Log 空间操作) ---
# PACF 告诉我们 1, 2, 24 是核心
for lag in [1, 2, 24, 48, 72, 168]:
    df[f'lag_{lag}'] = df['target_log'].shift(lag)

df = df.dropna()
df.head()

## 2.3 模型表现评估

评估指标：MAE、MSE、MAPE、RMSE。

In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    mean_absolute_percentage_error, root_mean_squared_error
)

def perf_metrics(y_true, y_pred):
    result = []
    for func in [mean_absolute_error, mean_squared_error,
                 mean_absolute_percentage_error, root_mean_squared_error]:
        result.append(func(y_true, y_pred))
    return pd.Series(result, index=["MAE", "MSE", "MAPE", "RMSE"])

## 2.4 基准模型比较 — LR vs XGBoost

采用**逐步扩大训练集**的滚动验证（walk-forward）：以每个 `test_year` 之前的所有数据为训练集，预测该年；最后评估指标取各年均值。

In [ ]:
df_final = df.copy()

# 1. 定义不同模型的特征
lr_features = [
    'hour_sin', 'hour_cos', 'hour_sin_weekend', 'hour_cos_weekend',
    'is_peak_month', 'is_peak_hour', 'is_weekend', 'is_holiday', 'is_working_hour',
    'lag_1', 'lag_2', 'lag_24', 'lag_48', 'lag_72', 'lag_168',
    'rolling_mean_24', 'lag_diff'
]

xgb_features = [
    'hour', 'month', 'quarter', 'year', 'dayofweek', 'is_weekend', 'hour_day_idx',
    'is_peak_month', 'is_peak_hour', 'is_holiday', 'is_working_hour',
    'lag_1', 'lag_2', 'lag_24', 'lag_48', 'lag_72', 'lag_168',
    'rolling_mean_24', 'lag_diff'
]

# 2. 验证循环
test_years = range(2013, 2019)
results = {}
models = {}
preds = {}

for test_year in test_years:
    train_data = df_final[df_final.index.year < test_year]
    test_data = df_final[df_final.index.year == test_year]

    y_train = train_data['target_log']
    y_test_log = test_data['target_log']
    y_test_real = test_data['amount'].values  # 真实值用于最后衡量误差

    # --- 模型 A: Linear Regression (statsmodels OLS) ---
    X_train_lr = train_data[lr_features].values
    X_train_lr_with_const = sm.add_constant(X_train_lr)
    lr_model = sm.OLS(y_train, X_train_lr_with_const)
    lr_results = lr_model.fit()  # 丰富的 Result 对象
    lr_pred_log = lr_results.predict(sm.add_constant(test_data[lr_features].values))
    lr_pred_real = np.expm1(lr_pred_log)  # 还原回原始量纲

    # --- 模型 B: XGBoost ---
    xgb_model = xgb.XGBRegressor(
        n_estimators=1000, learning_rate=0.05, max_depth=5,
        n_jobs=-1, early_stopping_rounds=50
    )
    xgb_model.fit(
        train_data[xgb_features].values, y_train.values,
        eval_set=[(test_data[xgb_features].values, y_test_log.values)],
        verbose=False
    )
    xgb_pred_log = xgb_model.predict(test_data[xgb_features].values)
    xgb_pred_real = np.expm1(xgb_pred_log)

    # 记录结果
    lr_metrics = perf_metrics(y_test_real, lr_pred_real).rename("lr")
    xgb_metrics = perf_metrics(y_test_real, xgb_pred_real).rename("xgb")

    results[str(test_year)] = pd.concat([lr_metrics, xgb_metrics], axis=1)
    models[str(test_year)] = {"lr": lr_results, "xgb": xgb_model}
    preds[str(test_year)] = pd.DataFrame({
        "real": y_test_real,
        "lr": lr_pred_real,
        "xgb": xgb_pred_real
    }, index=test_data.index)
    print(f"Year {test_year} Finished")

### 结果分析

可以从以下角度分析：
1. 平均评估指标
2. 平均 LR 系数、p 值 与 平均 XGBoost 特征重要性
3. 拼接预测值，按月 / 按 weekday groupby 看评估指标 —— 是否仍存在季节/小时效应
4. 预测残差分布 (hist + skew/kurt)
5. 随机抽样一段时间画真实值 vs 预测值
6. 检测残差序列是否为白噪声

In [ ]:
# 平均评估指标
results = pd.concat(results.values(), keys=results.keys(), axis=0).unstack(level=1)
results.mean(axis=0).to_frame().unstack()

In [ ]:
# LR 的所有特征系数与 p 值
def get_year_params_lr(year: int):
    lr_model = models[str(year)]['lr']
    coef_df = pd.DataFrame({'Coef': lr_model.params, 'P-Value': lr_model.pvalues})
    coef_df.index = ['const'] + lr_features
    print(f"{year} LR R-square: {lr_model.rsquared: .4f}")
    return coef_df

coef_df = {}
for year in range(2013, 2019):
    coef_df[str(year)] = get_year_params_lr(year)
coef_df = pd.concat(coef_df.values(), keys=coef_df.keys(), axis=1).stack(level=1, future_stack=True)
coef_df.mean(axis=1).unstack(level=1).sort_values(by="P-Value")

In [ ]:
# XGB 的特征重要性
def get_year_xgb_importance(year: int):
    xgb_model = models[str(year)]['xgb']
    importance = pd.DataFrame({
        'Feature': xgb_features,
        'Importance': xgb_model.feature_importances_
    }).set_index("Feature")
    return importance

importance = {}
for year in range(2013, 2019):
    importance[str(year)] = get_year_xgb_importance(year)
importance = pd.concat(importance.values(), keys=importance.keys(), axis=1).droplevel(1, axis=1)
importance.mean(axis=1).sort_values(ascending=False)

In [ ]:
# 拼接预测值，计算残差
preds = pd.concat(preds.values(), axis=0).sort_index()
preds["lr_residuals"] = preds['real'].sub(preds['lr'])
preds['xgb_residuals'] = preds['real'].sub(preds['xgb'])

# 残差分布
plt.figure(figsize=(10, 4))
sns.histplot(preds['lr_residuals'], kde=True, label="LR Residuals")
sns.histplot(preds['xgb_residuals'], kde=True, label="XGB Residuals")
print(preds.filter(like="residuals").agg(["skew", "kurt"]))
plt.legend()
plt.show()

In [ ]:
# 按月计算评估指标并绘图
by_month_perf_lr = preds.groupby(preds.index.month).apply(lambda x: perf_metrics(x["real"], x["lr"]))
by_month_perf_xgb = preds.groupby(preds.index.month).apply(lambda x: perf_metrics(x["real"], x["xgb"]))
by_month_perf = pd.concat([by_month_perf_lr, by_month_perf_xgb], axis=1, keys=['lr', 'xgb']).swaplevel(0, 1, axis=1).sort_index(axis=1)

fig, axs = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
axs = axs.flatten()
for i, col in enumerate(by_month_perf.columns.get_level_values(0).unique()):
    sns.lineplot(by_month_perf[col], ax=axs[i])
    axs[i].set_title(col)
plt.suptitle("Performance by Month")
plt.show()

In [ ]:
# 按小时计算评估指标并绘图
by_hour_perf_lr = preds.groupby(preds.index.hour).apply(lambda x: perf_metrics(x["real"], x["lr"]))
by_hour_perf_xgb = preds.groupby(preds.index.hour).apply(lambda x: perf_metrics(x["real"], x["xgb"]))
by_hour_perf = pd.concat([by_hour_perf_lr, by_hour_perf_xgb], axis=1, keys=['lr', 'xgb']).swaplevel(0, 1, axis=1).sort_index(axis=1)

fig, axs = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
axs = axs.flatten()
for i, col in enumerate(by_hour_perf.columns.get_level_values(0).unique()):
    sns.lineplot(by_hour_perf[col], ax=axs[i])
    axs[i].set_title(col)
plt.suptitle("Performance by Hour")
plt.show()

In [ ]:
# 随机抽样查看预测值（日度均值）
pred_daily = preds.iloc[:, :3].resample("D", label="right", closed="right").mean()

start = np.random.choice(pred_daily.shape[0] - 30)

plt.figure(figsize=(15, 6))
sns.lineplot(pred_daily.iloc[start:start+30], ax=plt.gca(), alpha=0.7)
plt.title("Sampled 30-Day Window: Real vs Predictions")
plt.legend()
plt.show()

In [ ]:
# 检验残差是否为白噪声 (Ljung-Box)
from statsmodels.stats.diagnostic import acorr_ljungbox

lb_test_lr = acorr_ljungbox(preds['lr_residuals'], lags=[10, 24], return_df=True)
print("LR residuals:")
print(lb_test_lr)
lb_test_xgb = acorr_ljungbox(preds['xgb_residuals'], lags=[10, 24], return_df=True)
print("\nXGB residuals:")
print(lb_test_xgb)

## 2.5 最终模型 — XGBoost 参数优化

用 `RandomizedSearchCV` + `TimeSeriesSplit` 进行时间序列友好的超参搜索。

> 注：参数搜索较耗时，可调小 `n_iter` 或缩减 `param_grid`。

In [ ]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

df_final = df.copy()
xgb_features = [
    'hour', 'month', 'quarter', 'year', 'dayofweek', 'is_weekend', 'hour_day_idx',
    'is_peak_month', 'is_peak_hour', 'is_holiday', 'is_working_hour',
    'lag_1', 'lag_2', 'lag_24', 'lag_48', 'lag_72', 'lag_168',
    'rolling_mean_24', 'lag_diff'
]

# 1. 划分训练集与测试集
train_mask = df_final.index.year <= 2014
test_mask = df_final.index.year >= 2015

X_train = df_final.loc[train_mask, xgb_features]
y_train_log = df_final.loc[train_mask, 'target_log']

X_test = df_final.loc[test_mask, xgb_features]
y_test_real = df_final.loc[test_mask, 'amount']  # 真实值用于最后评估

# 2. 定义参数搜索空间
param_grid = {
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [500, 1000, 1500],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'reg_lambda': [1, 5, 10],
    'gamma': [0, 0.1, 0.2]
}

# 3. 时间序列交叉验证
tscv = TimeSeriesSplit(n_splits=5)

# 4. 随机搜索
random_search = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(objective='reg:squarederror', tree_method='hist', n_jobs=-1),
    param_distributions=param_grid,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_squared_error',
    verbose=1,
    random_state=42
)

# 5. 执行搜索
print("开始参数搜索...")
random_search.fit(X_train, y_train_log)

# 6. 最优模型
best_xgb = random_search.best_estimator_
print(f"最优参数: {random_search.best_params_}")

# 7. 测试集预测
y_pred_log = best_xgb.predict(X_test)
y_pred_real = np.expm1(y_pred_log)

# 8. 性能评估
metrics = perf_metrics(y_test_real, y_pred_real)
metrics

> 最终模型的结果同样可以参照 **2.4** 部分进行评估（残差分布、按月/按小时指标、白噪声检验等）。

# 3. 最终结论

总结数据探索的发现、模型训练与预测结果，反思可能提升预测准确度的其他方法：

- **数据探索发现**：长期趋势（year）、稳定的年内季节性（month）、清晰的周内 + 小时双峰效应、节假日下行效应、以及历史上的异常段。
- **模型结果**：XGBoost 通常优于 LR（能捕捉非线性与交叉效应）；lag_1 / lag_24 / lag_168 与 rolling_mean_24 往往是最重要的特征。
- **改进方向**：
  - 按月 / 按小时的残差若仍存在周期，说明仍有未被建模的季节/小时效应 —— **减去历史同一周期的 mean**（去周期 / deseasonalize）后再建模，可能有效。
  - 加入更多滞后与滚动统计（如 rolling std、不同窗口）。
  - 对节假日前后、特殊时段做更精细的 dummy。
  - 残差非白噪声 → 考虑 ARIMA 残差建模或 hybrid 方法。